In [ ]:
!pip install -U fastai "fastcore<2" torchinfo grad-cam statsmodels -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
TRAINING_VALIDATION_ROOT = Path("/content/drive/MyDrive/Masters/Datasets/train_and_validation_sets")
TEST_ROOT = Path("/content/drive/MyDrive/Masters/Datasets/test_set")

RESULTS_DIR = Path("/content/drive/MyDrive/Masters/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def make_df(root):
    records = []
    for mayo_dir in sorted(root.iterdir()):
        if not mayo_dir.is_dir():
            continue
        label = mayo_dir.name
        for img_path in mayo_dir.glob("*.bmp"):
            patient_id = img_path.stem.split("_")[2]
            records.append({"path": str(img_path), "label": label, "patient_id": patient_id})
    return pd.DataFrame(records)

training_validation_df = make_df(TRAINING_VALIDATION_ROOT)
test_df = make_df(TEST_ROOT)

mayo_grades = sorted(training_validation_df["label"].unique())

training_validation_counts = training_validation_df["label"].value_counts().reindex(mayo_grades, fill_value=0)
test_counts = test_df["label"].value_counts().reindex(mayo_grades, fill_value=0)

summary_df = pd.DataFrame({
    "MES Grade": mayo_grades,
    "Train + Val": training_validation_counts.values,
    "Test": test_counts.values,
})
summary_df["Total"] = summary_df["Train + Val"] + summary_df["Test"]

total_row = pd.DataFrame([{
    "MES Grade": "Total",
    "Train + Val": int(summary_df["Train + Val"].sum()),
    "Test": int(summary_df["Test"].sum()),
    "Total": int(summary_df["Total"].sum()),
}])
summary_df = pd.concat([summary_df, total_row], ignore_index=True)

summary_df

In [ ]:
import gc

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score as sklearn_f1
from sklearn.metrics import cohen_kappa_score
from torchvision.models import mobilenet_v2, densenet121, efficientnet_b0
from fastai.vision.all import (
    ImageDataLoaders, Resize, aug_transforms, Normalize, imagenet_stats,
    vision_learner, resnet34, valley, accuracy, F1Score, CohenKappa, load_learner,
)

In [ ]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
patient_dominant_label_list = []

for patient_id in training_validation_df["patient_id"].unique():
    patient_rows = training_validation_df[training_validation_df["patient_id"] == patient_id]
    label_counts = patient_rows["label"].value_counts()
    most_common_label = label_counts.index[0]
    patient_dominant_label_list.append({"patient_id": patient_id, "dominant_label": most_common_label})

patient_dominant_label = pd.DataFrame(patient_dominant_label_list)

patients = patient_dominant_label["patient_id"].values
strata_labels = patient_dominant_label["dominant_label"].values

training_patients, validation_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42,
    stratify=strata_labels,
)

is_valid_list = []
for patient_id in training_validation_df["patient_id"]:
    if patient_id in validation_patients:
        is_valid_list.append(True)
    else:
        is_valid_list.append(False)
training_validation_df["is_valid"] = is_valid_list

print(f"Training patients: {len(training_patients)}")
print(f"Validation patients: {len(validation_patients)}")
print(f"Training images: {training_validation_df['is_valid'].eq(False).sum()}")
print(f"Validation images: {training_validation_df['is_valid'].eq(True).sum()}")

In [ ]:
training_split_df = training_validation_df[training_validation_df["is_valid"] == False]
validation_split_df = training_validation_df[training_validation_df["is_valid"] == True]

print("Train split class distribution:")
print(training_split_df["label"].value_counts().sort_index())
print("\nValidation split class distribution:")
print(validation_split_df["label"].value_counts().sort_index())

training_patients_check = training_split_df["patient_id"].unique()
validation_patients_check = validation_split_df["patient_id"].unique()

overlap_found = False
for patient_id in training_patients_check:
    if patient_id in validation_patients_check:
        overlap_found = True

if overlap_found:
    raise ValueError("Patient leakage detected between train and validation splits!")
else:
    print("\nNo patient overlap between train and validation splits.")

In [ ]:
import os

IMG_SIZE = 224
BATCH_SIZE = 32


augmentations = aug_transforms(
    flip_vert=False,
    max_rotate=15,
    max_zoom=1.1,
    max_lighting=0.2,
    p_lighting=0.75,
)
mean, std = imagenet_stats
normalization = Normalize.from_stats(mean, std)
batch_transforms = augmentations + [normalization]

dls = ImageDataLoaders.from_df(
    training_validation_df,
    path="/",
    fn_col="path",
    label_col="label",
    is_valid_col="is_valid",
    item_tfms=Resize(IMG_SIZE),
    batch_tfms=batch_transforms,
    bs=BATCH_SIZE,
    num_workers=2,
)
dls.path = TRAINING_VALIDATION_ROOT.parent

print("Class vocabulary:", dls.vocab)
print("Training set size:", len(dls.train_ds))
print("Validation set size:", len(dls.valid_ds))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

vocab = dls.vocab
C = len(vocab)
counts_ordered = training_validation_df["label"].value_counts().reindex(vocab, fill_value=0)
weights = compute_class_weight(class_weight="balanced", classes=np.array(vocab), y=training_validation_df["label"])
weights = weights / weights.sum() * C

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Running on: {device}")
for cls, n, w in zip(vocab, counts_ordered, weights):
    print(f"  {cls}: count={n:>5d}  weight={w:.4f}")
weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
weighted_ce_loss = nn.CrossEntropyLoss(weight=weights_tensor, reduction="mean")

In [ ]:
ARCHITECTURES = {
    "ResNet34": resnet34,
    "MobileNetV2": mobilenet_v2,
    "DenseNet121": densenet121,
    "EfficientNet-B0": efficientnet_b0,
}

from fastai.vision.learner import model_meta, _default_meta
model_meta[mobilenet_v2] = {**_default_meta, "cut": -1}

EPOCHS_HEAD = 5
EPOCHS_FULL = 15

print(f"Architectures: {list(ARCHITECTURES)}  epochs: {EPOCHS_HEAD}+{EPOCHS_FULL}")

In [ ]:
import time
# Source: https://github.com/MaximeGloesener/torch-benchmark
# Source https://docs.pytorch.org/docs/2.14/generated/torch.cuda.synchronize.html

def benchmark_latency(learn, dls, n_warmup=10, n_repeats=100):
    model = learn.model
    model.eval()
    original_device = next(model.parameters()).device

    xb, _ = dls.valid.one_batch()
    xb = xb[:1]

    def _time_on(device):
        model.to(device)
        x = xb.to(device)
        is_cuda = device.type == "cuda"

        with torch.no_grad():
            for _ in range(n_warmup):
                _ = model(x)
            if is_cuda:
                torch.cuda.synchronize()

            times_ms = []
            for _ in range(n_repeats):
                if is_cuda:
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                _ = model(x)
                if is_cuda:
                    torch.cuda.synchronize()
                times_ms.append((time.perf_counter() - t0) * 1000)

        return float(np.mean(times_ms)), float(np.std(times_ms))

    cpu_ms_mean, cpu_ms_std = _time_on(torch.device("cpu"))

    gpu_device = torch.device("cuda") if torch.cuda.is_available() else None

    if gpu_device is not None:
        gpu_ms_mean, gpu_ms_std = _time_on(gpu_device)
    else:
        gpu_ms_mean, gpu_ms_std = None, None

    model.to(original_device)

    if gpu_ms_mean is not None:
        gpu_str = f"{gpu_ms_mean:.2f} ± {gpu_ms_std:.2f} ms/image"
    else:
        gpu_str = "N/A (no CUDA available)"
    print(f"LATENCY -> CPU: {cpu_ms_mean:.2f} ± {cpu_ms_std:.2f} ms/image, GPU: {gpu_str}")

    return {
        "cpu_ms_mean": round(cpu_ms_mean, 4),
        "cpu_ms_std": round(cpu_ms_std, 4),
        "gpu_ms_mean": round(gpu_ms_mean, 4) if gpu_ms_mean is not None else None,
        "gpu_ms_std": round(gpu_ms_std, 4) if gpu_ms_std is not None else None,
    }

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

def compute_gradcam(learn, image_tensor, target_class_idx):
    model = learn.model
    model.eval()
    device = next(model.parameters()).device

    if image_tensor.dim() == 3:
        x = image_tensor.unsqueeze(0)
    else:
        x = image_tensor
    x = x.to(device)

    backbone = model[0]
    target_layer = None
    for module in backbone.modules():
        if isinstance(module, nn.Conv2d):
            target_layer = module

    if target_layer is None:
        raise RuntimeError("No Conv2d layer found in backbone")

    targets = [ClassifierOutputTarget(target_class_idx)]
    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        heatmap = cam(input_tensor=x, targets=targets)

    return heatmap[0]

def select_shared_gradcam_images(test_df, dls, true_cls, correct_counts, n_architectures):
    vocab = list(dls.vocab)
    selected = []
    for class_idx, class_name in enumerate(vocab):
        class_indices = []
        for idx in range(len(true_cls)):
            if true_cls[idx] == class_idx:
                class_indices.append(idx)

        chosen_idx = None
        for threshold in reversed(range(n_architectures + 1)):
            candidates = []
            for idx in class_indices:
                if correct_counts[idx] >= threshold:
                    candidates.append(idx)
            if len(candidates) > 0:
                chosen_idx = candidates[0]
                if threshold < n_architectures:
                    print(f"[gradcam] WARNING: no {class_name} image correctly classified by all "
                          f"{n_architectures} architectures - using one correctly classified by >= {threshold} instead")
                break

        path = test_df["path"].iloc[chosen_idx]
        selected.append({"class_idx": class_idx, "class_name": class_name, "path": path})
    return selected

def visualize_interpretability(learn, dls, gradcam_images, architecture_name):
    slug = architecture_name.replace(" ", "_").replace("-", "_").lower()
    out_dir = RESULTS_DIR / slug
    out_dir.mkdir(parents=True, exist_ok=True)

    device = next(learn.model.parameters()).device
    vocab = list(dls.vocab)
    selected = []

    for item in gradcam_images:
        img_path = item["path"]
        xb = dls.test_dl([img_path], num_workers=0).one_batch()[0]
        with torch.no_grad():
            pred = learn.model(xb.to(device))
        pred_indices = pred.argmax(dim=1)
        pred_idx = int(pred_indices[0])

        cam = compute_gradcam(learn, xb[0], pred_idx)
        orig_img = np.array(Image.open(img_path).convert("RGB").resize((224, 224)))

        selected.append({
            "class_name": item["class_name"],
            "path": img_path,
            "pred_idx": pred_idx,
            "orig_img": orig_img,
            "cam": cam,
        })

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for col, item in enumerate(selected):
        pred_label = vocab[item["pred_idx"]]

        axes[0, col].imshow(item["orig_img"])
        axes[0, col].set_title(f"True: {item['class_name']}\nPred: {pred_label}", fontsize=10)
        axes[0, col].axis("off")

        orig_img_float = item["orig_img"].astype(np.float32) / 255.0
        overlay = show_cam_on_image(orig_img_float, item["cam"], use_rgb=True, image_weight=0.55)
        axes[1, col].imshow(overlay)
        axes[1, col].set_title("Grad-CAM", fontsize=10)
        axes[1, col].axis("off")

    plt.tight_layout()
    save_path = out_dir / "gradcam.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Grad-CAM figure saved -> {save_path}")

    return [(item["path"], item["pred_idx"]) for item in selected]

In [ ]:
def train_architecture(architecture_name, architecture_function, dls, loss_function):
    print(f"\n{'=' * 60}\n  Architecture: {architecture_name}\n{'=' * 60}")

    qwk = CohenKappa(weights="quadratic")
    f1 = F1Score(average="macro")
    metrics_list = [accuracy, f1, qwk]

    learn = vision_learner(dls, architecture_function, loss_func=loss_function, metrics=metrics_list)

    print("\n-- Phase 1: frozen backbone --")
    lr_head = learn.lr_find(suggest_funcs=(valley,), show_plot=False).valley
    print(f"  LR (valley): {lr_head:.2e}")
    learn.fit_one_cycle(EPOCHS_HEAD, lr_head)

    print("\n-- Phase 2: full fine-tune --")
    learn.unfreeze()
    lr_full = learn.lr_find(suggest_funcs=(valley,), show_plot=False).valley
    print(f"  LR (valley): {lr_full:.2e}")
    learn.fit_one_cycle(EPOCHS_FULL, slice(lr_full / 10, lr_full))

    final = learn.recorder.values[-1]
    metrics = {
        "training_loss": float(final[0]),
        "validation_loss": float(final[1]),
        "validation_accuracy": float(final[2]),
        "validation_f1_macro": float(final[3]),
        "validation_qwk": float(final[4]),
    }
    print(f"\n  Done -> validation_accuracy={metrics['validation_accuracy']:.4f}  "
          f"f1={metrics['validation_f1_macro']:.4f}  qwk={metrics['validation_qwk']:.4f}")
    return learn, metrics

In [ ]:
def evaluate_on_test(learn, test_df, dls):
    test_dl = dls.test_dl(test_df["path"].tolist(), num_workers=2)
    test_preds_raw, _ = learn.get_preds(dl=test_dl)

    label_to_idx = {v: i for i, v in enumerate(dls.vocab)}
    test_targets = torch.tensor([label_to_idx[l] for l in test_df["label"]])

    pred_cls = test_preds_raw.argmax(dim=1).numpy()
    true_cls = test_targets.numpy()

    test_acc = float((pred_cls == true_cls).mean())
    test_f1 = float(sklearn_f1(true_cls, pred_cls, average="macro"))
    test_qwk = float(cohen_kappa_score(true_cls, pred_cls, weights="quadratic"))

    print(f"  TEST -> acc={test_acc:.4f}  f1={test_f1:.4f}  qwk={test_qwk:.4f}")
    return {"test_acc": test_acc, "test_f1": test_f1, "test_qwk": test_qwk}, pred_cls, true_cls

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

In [ ]:
from scipy.stats import bootstrap

def bootstrap_ci(true_cls, pred_cls, n_resamples=1000, random_state=42):
    rng = np.random.default_rng(random_state)

    def _acc_stat(t, p):
        return (p == t).mean()

    def _f1_stat(t, p):
        return sklearn_f1(t, p, average="macro")

    def _qwk_stat(t, p):
        try:
            return cohen_kappa_score(t, p, weights="quadratic")
        except ValueError:
            return np.nan

    acc_res = bootstrap(
        (true_cls, pred_cls), _acc_stat,
        n_resamples=n_resamples, paired=True, vectorized=False,
        method="percentile", confidence_level=0.95, random_state=rng,
    )
    f1_res = bootstrap(
        (true_cls, pred_cls), _f1_stat,
        n_resamples=n_resamples, paired=True, vectorized=False,
        method="percentile", confidence_level=0.95, random_state=rng,
    )
    qwk_res = bootstrap(
        (true_cls, pred_cls), _qwk_stat,
        n_resamples=n_resamples, paired=True, vectorized=False,
        method="percentile", confidence_level=0.95, random_state=rng,
    )

    qwk_low, qwk_high = np.nanpercentile(qwk_res.bootstrap_distribution, [2.5, 97.5])

    ci = {
        "acc_ci_low": float(acc_res.confidence_interval.low),
        "acc_ci_high": float(acc_res.confidence_interval.high),
        "f1_ci_low": float(f1_res.confidence_interval.low),
        "f1_ci_high": float(f1_res.confidence_interval.high),
        "qwk_ci_low": float(qwk_low),
        "qwk_ci_high": float(qwk_high),
    }

    print(f"BOOTSTRAP CI (95%) -> acc: [{ci['acc_ci_low']:.4f}, {ci['acc_ci_high']:.4f}], "
          f"f1: [{ci['f1_ci_low']:.4f}, {ci['f1_ci_high']:.4f}], "
          f"qwk: [{ci['qwk_ci_low']:.4f}, {ci['qwk_ci_high']:.4f}]")

    return ci

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def run_mcnemar_test(preds_a, preds_b, name_a="A", name_b="B"):
    assert np.array_equal(preds_a["true"], preds_b["true"]), \
        "Both models must be evaluated on the same test set (true labels differ)"

    true = preds_a["true"]
    correct_a = preds_a["pred"] == true
    correct_b = preds_b["pred"] == true

    both_correct      = int((correct_a & correct_b).sum())
    a_correct_b_wrong = int((correct_a & ~correct_b).sum())
    a_wrong_b_correct = int((~correct_a & correct_b).sum())
    both_wrong        = int((~correct_a & ~correct_b).sum())

    table = [[both_correct, a_correct_b_wrong],
             [a_wrong_b_correct, both_wrong]]

    n_disagreements = a_correct_b_wrong + a_wrong_b_correct
    is_exact = n_disagreements < 25

    result = mcnemar(table, exact=is_exact, correction=not is_exact)

    print(f"McNemar {name_a} vs {name_b}: p={result.pvalue:.4f} (n_disagreements={n_disagreements})")

    return {
        "statistic": float(result.statistic),
        "p_value": float(result.pvalue),
        "n_disagreements": n_disagreements,
    }

In [ ]:
all_results = []
stored_preds = {}
stored_correct_indices = {}

for architecture_name, architecture_function in ARCHITECTURES.items():
    learn, training_metrics = train_architecture(architecture_name, architecture_function, dls, weighted_ce_loss)
    test_metrics, pred_cls, true_cls = evaluate_on_test(learn, test_df, dls)
    ci = bootstrap_ci(true_cls, pred_cls)
    stored_preds[architecture_name] = {"pred": pred_cls, "true": true_cls}
    stored_correct_indices[architecture_name] = np.where(pred_cls == true_cls)[0]
    latency_metrics = benchmark_latency(learn, dls)

    arch_slug = architecture_name.replace(" ", "_").replace("-", "_").lower()
    learner_path = RESULTS_DIR / f"{arch_slug}_learner.pkl"
    learn.export(fname=learner_path)
    print(f"Learner saved -> {learner_path}")

    learn.recorder.plot_loss()
    plt.show()

    all_results.append({
        "model": architecture_name,
        "validation_accuracy": round(training_metrics["validation_accuracy"], 4),
        "validation_f1_macro": round(training_metrics["validation_f1_macro"], 4),
        "validation_qwk": round(training_metrics["validation_qwk"], 4),
        "test_accuracy": round(test_metrics["test_acc"], 4),
        "test_f1_macro": round(test_metrics["test_f1"], 4),
        "test_qwk": round(test_metrics["test_qwk"], 4),
        "acc_ci_low": round(ci["acc_ci_low"], 4),
        "acc_ci_high": round(ci["acc_ci_high"], 4),
        "f1_ci_low": round(ci["f1_ci_low"], 4),
        "f1_ci_high": round(ci["f1_ci_high"], 4),
        "qwk_ci_low": round(ci["qwk_ci_low"], 4),
        "qwk_ci_high": round(ci["qwk_ci_high"], 4),
        "cpu_ms_mean": latency_metrics["cpu_ms_mean"],
        "cpu_ms_std": latency_metrics["cpu_ms_std"],
        "gpu_ms_mean": latency_metrics["gpu_ms_mean"],
        "gpu_ms_std": latency_metrics["gpu_ms_std"],
    })

    pd.DataFrame(all_results).set_index("model").to_csv(RESULTS_DIR / "final_results.csv")
    print(f"[checkpoint] saved {len(all_results)}/{len(ARCHITECTURES)} architectures -> {RESULTS_DIR / 'final_results.csv'}")

    del learn
    clear_memory()

results_df = pd.DataFrame(all_results).set_index("model")
results_df = results_df.sort_values("test_qwk", ascending=False)

from IPython.display import display

def _format_ci(low, high):
    return f"[{low:.4f}, {high:.4f}]"

def _format_mean_std(mean, std):
    if mean is None or std is None:
        return "N/A"
    return f"{mean:.2f} ± {std:.2f}"

perf_table = pd.DataFrame({
    "Accuracy": results_df["test_accuracy"].round(4),
    "Acc 95% CI": [_format_ci(lo, hi) for lo, hi in zip(results_df["acc_ci_low"], results_df["acc_ci_high"])],
    "F1 (macro)": results_df["test_f1_macro"].round(4),
    "F1 95% CI": [_format_ci(lo, hi) for lo, hi in zip(results_df["f1_ci_low"], results_df["f1_ci_high"])],
    "QWK": results_df["test_qwk"].round(4),
    "QWK 95% CI": [_format_ci(lo, hi) for lo, hi in zip(results_df["qwk_ci_low"], results_df["qwk_ci_high"])],
}, index=results_df.index)

validation_table = pd.DataFrame({
    "Validation Accuracy": results_df["validation_accuracy"].round(4),
    "Validation F1 (macro)": results_df["validation_f1_macro"].round(4),
    "Validation QWK": results_df["validation_qwk"].round(4),
}, index=results_df.index)

latency_table = pd.DataFrame({
    "CPU (ms/image)": [_format_mean_std(m, s) for m, s in zip(results_df["cpu_ms_mean"], results_df["cpu_ms_std"])],
    "GPU (ms/image)": [_format_mean_std(m, s) for m, s in zip(results_df["gpu_ms_mean"], results_df["gpu_ms_std"])],
}, index=results_df.index)

_table_styles = [
    {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "6px")]},
    {"selector": "th", "props": [("text-align", "center"), ("padding", "4px 12px")]},
    {"selector": "td", "props": [("text-align", "center"), ("padding", "4px 12px")]},
]

display(
    perf_table.style
    .format({"Accuracy": "{:.4f}", "F1 (macro)": "{:.4f}", "QWK": "{:.4f}"})
    .set_caption("Performance Metrics (Test Set)")
    .set_table_styles(_table_styles)
)
perf_table.to_csv(RESULTS_DIR / "performance_metrics.csv")
print(f"Saved -> {RESULTS_DIR / 'performance_metrics.csv'}")

display(
    validation_table.style
    .format({"Validation Accuracy": "{:.4f}", "Validation F1 (macro)": "{:.4f}", "Validation QWK": "{:.4f}"})
    .set_caption("Validation Metrics")
    .set_table_styles(_table_styles)
)
validation_table.to_csv(RESULTS_DIR / "validation_metrics.csv")
print(f"Saved -> {RESULTS_DIR / 'validation_metrics.csv'}")

display(
    latency_table.style
    .set_caption("Latency (Inference Cost)")
    .set_table_styles(_table_styles)
)
latency_table.to_csv(RESULTS_DIR / "latency_metrics.csv")
print(f"Saved -> {RESULTS_DIR / 'latency_metrics.csv'}")

In [ ]:
n_architectures = len(ARCHITECTURES)

correct_counts = np.zeros(len(test_df), dtype=int)
for indices in stored_correct_indices.values():
    correct_counts[indices] += 1

true_cls_shared = next(iter(stored_preds.values()))["true"]
gradcam_images = select_shared_gradcam_images(test_df, dls, true_cls_shared, correct_counts, n_architectures)
print("Shared grad-cam images (one per mayo grade):")
for item in gradcam_images:
    print(f"  {item['class_name']}: {item['path']}")

gradcam_selected_by_arch = {}
for architecture_name in ARCHITECTURES:
    arch_slug = architecture_name.replace(" ", "_").replace("-", "_").lower()
    learn = load_learner(RESULTS_DIR / f"{arch_slug}_learner.pkl")
    gradcam_selected_by_arch[architecture_name] = visualize_interpretability(learn, dls, gradcam_images, architecture_name)
    del learn
    clear_memory()

In [ ]:
baseline_name = "ResNet34"

mcnemar_results = []
for architecture_name, preds in stored_preds.items():
    if architecture_name == baseline_name:
        continue
    result = run_mcnemar_test(stored_preds[baseline_name], preds, name_a=baseline_name, name_b=architecture_name)
    mcnemar_results.append({
        "comparison": f"{baseline_name} vs {architecture_name}",
        "p_value": round(result["p_value"], 4),
        "n_disagreements": result["n_disagreements"],
        "significant": result["p_value"] < 0.05,
    })

mcnemar_df = pd.DataFrame(mcnemar_results)
print("\nMcNEMAR PAIRWISE COMPARISONS (baseline: ResNet34):")
print(mcnemar_df.to_string(index=False))

mcnemar_out_path = RESULTS_DIR / "mcnemar_pairwise.csv"
mcnemar_out_path.parent.mkdir(parents=True, exist_ok=True)
mcnemar_df.to_csv(mcnemar_out_path, index=False)
print(f"\nSaved -> {mcnemar_out_path}")

In [ ]:
final_results_path = RESULTS_DIR / "final_results.csv"
results_df.to_csv(final_results_path)
print(f"Final results saved -> {final_results_path}")